 # Session 5.3 — Numeric Manipulation & Linear Algebra Basics



1. Array reshaping — going deeper than Session 
2. Transposition — flipping rows and columns
3. Matrix multiplication basics — what it means and how to do it
4. Generating synthetic data — building realistic datasets from scratch

---

> **Where we left off:** Over the last two sessions we learned why NumPy is fast, how arrays are structured, and how to do "no-loop" math using element-wise operations, broadcasting, aggregation, and masking. Today we add the remaining core skills: rearranging data into the shape you need (reshaping and transposition), combining two datasets through matrix multiplication, and — critically — generating realistic fake data for testing, prototyping, and learning. We continue as analysts for our IPL franchise.

In [1]:
import numpy as np

print("NumPy version:", np.__version__)
print("Ready.")

NumPy version: 2.3.5
Ready.


In [5]:
PLAYERS = ["Virat", "Rohit", "Hardik", "Shubman", "Shreyas"]

squad_scores = np.array([
    [72, 45, 88, 31, 64,  0, 103, 52, 19, 77, 41, 93, 28, 66],   # Virat
    [56,  0, 74, 88, 32, 45,  61, 17, 90, 38, 72,  5, 83, 49],   # Rohit
    [45, 32,  0, 67, 18, 54,  78, 22, 41, 33, 60, 15, 88, 27],   # Hardik
    [33, 61, 45,  0, 77, 83,  22, 58, 44, 71,  0, 66, 31, 52],   # Shubman
    [88, 14, 57, 72,  0, 38,  91, 43, 66, 22, 84,  0, 47, 73],   # Shreyas
])

print(f"squad_scores shape: {squad_scores.shape}")
print(squad_scores)

squad_scores shape: (5, 14)
[[ 72  45  88  31  64   0 103  52  19  77  41  93  28  66]
 [ 56   0  74  88  32  45  61  17  90  38  72   5  83  49]
 [ 45  32   0  67  18  54  78  22  41  33  60  15  88  27]
 [ 33  61  45   0  77  83  22  58  44  71   0  66  31  52]
 [ 88  14  57  72   0  38  91  43  66  22  84   0  47  73]]


In [6]:
PLAYERS

['Virat', 'Rohit', 'Hardik', 'Shubman', 'Shreyas']

In [7]:
squad_scores

array([[ 72,  45,  88,  31,  64,   0, 103,  52,  19,  77,  41,  93,  28,
         66],
       [ 56,   0,  74,  88,  32,  45,  61,  17,  90,  38,  72,   5,  83,
         49],
       [ 45,  32,   0,  67,  18,  54,  78,  22,  41,  33,  60,  15,  88,
         27],
       [ 33,  61,  45,   0,  77,  83,  22,  58,  44,  71,   0,  66,  31,
         52],
       [ 88,  14,  57,  72,   0,  38,  91,  43,  66,  22,  84,   0,  47,
         73]])


#  Array Reshaping, In Depth



## The one rule that governs everything

**Reshaping never changes the data or the number of elements — only how they are organised into dimensions.**

If an array has 70 elements, every reshape of it must also have exactly 70 elements. `(5, 14)`, `(14, 5)`, `(7, 10)`, `(70,)`, `(2, 5, 7)` — all valid, because `5*14 = 14*5 = 7*10 = 70 = 2*5*7`. But `(5, 13)` is not valid — `5*13 = 65 ≠ 70`.

In [8]:
print(f"squad_scores has {squad_scores.size} total elements, shape {squad_scores.shape}")
print()

# Valid reshapes — all preserve 70 elements
reshape_7x10 = squad_scores.reshape(7, 10)
reshape_2x5x7 = squad_scores.reshape(2, 5, 7)
reshape_flat  = squad_scores.reshape(70)

print(f"reshape(7, 10)    shape: {reshape_7x10.shape}   size: {reshape_7x10.size}")
print(f"reshape(2, 5, 7)  shape: {reshape_2x5x7.shape}   size: {reshape_2x5x7.size}")
print(f"reshape(70)       shape: {reshape_flat.shape}     size: {reshape_flat.size}")
print()

# Invalid reshape — wrong total size
try:
    bad = squad_scores.reshape(5, 13)
except ValueError as e:
    print(f"reshape(5, 13) FAILS:")
    print(f"  ValueError: {e}")

squad_scores has 70 total elements, shape (5, 14)

reshape(7, 10)    shape: (7, 10)   size: 70
reshape(2, 5, 7)  shape: (2, 5, 7)   size: 70
reshape(70)       shape: (70,)     size: 70

reshape(5, 13) FAILS:
  ValueError: cannot reshape array of size 70 into shape (5,13)


In [9]:
# Flattening — going from many dimensions down to 1
# Two ways to do this: .flatten() and .ravel()

flat_a = squad_scores.flatten()
flat_b = squad_scores.ravel()

print(f"squad_scores shape: {squad_scores.shape}")
print(f".flatten() shape:   {flat_a.shape}")
print(f".ravel() shape:     {flat_b.shape}")
print()
print("Both give the same result here. The difference:")
print("  .flatten() always returns a NEW COPY of the data")
print("  .ravel()   returns a VIEW when possible (faster, shares memory)")
print()

# Demonstrating the difference
flat_copy = squad_scores.flatten()
flat_view = squad_scores.ravel()

flat_copy[0] = 999   # modify the flattened copy
flat_view[0] = 888   # modify the flattened view

print(f"After modifying flat_copy[0]: original squad_scores[0,0] = {squad_scores[0,0]}  (unchanged)")
print(f"After modifying flat_view[0]: original squad_scores[0,0] = {squad_scores[0,0]}  (changed! shares memory)")
print()
print("Rule of thumb: use .flatten() when you want a safe, independent copy.")
print("Use .ravel() only when you understand it may alias the original — or just use .flatten() by default.")

squad_scores shape: (5, 14)
.flatten() shape:   (70,)
.ravel() shape:     (70,)

Both give the same result here. The difference:
  .flatten() always returns a NEW COPY of the data
  .ravel()   returns a VIEW when possible (faster, shares memory)

After modifying flat_copy[0]: original squad_scores[0,0] = 888  (unchanged)
After modifying flat_view[0]: original squad_scores[0,0] = 888  (changed! shares memory)

Rule of thumb: use .flatten() when you want a safe, independent copy.
Use .ravel() only when you understand it may alias the original — or just use .flatten() by default.


In [10]:
# Oops — let's undo that accidental mutation from the demo above before continuing
squad_scores[0, 0] = 72
print("Restored:", squad_scores[0])

Restored: [ 72  45  88  31  64   0 103  52  19  77  41  93  28  66]


 ## Why reshape matters in practice

Reshaping is not just a party trick — it solves real problems:

- **Splitting a season into halves** — turning `(5, 14)` into `(5, 2, 7)` to compare first-half vs second-half form
- **Preparing data for a specific function** — many NumPy and machine learning functions expect a particular shape (e.g. images flattened to 1D, or a single sample reshaped to look like a batch of one)
- **Converting a flat list of measurements into a structured table**

In [11]:
# Practical example: split the season into two halves to compare form

# squad_scores shape (5, 14) -> reshape to (5, 2, 7)
# axis 0 = players, axis 1 = half (0=first half, 1=second half), axis 2 = match within that half
season_halves = squad_scores.reshape(5, 2, 7)

print(f"Reshaped to: {season_halves.shape}")
print()

first_half  = season_halves[:, 0, :]   # all players, first half, all 7 matches
second_half = season_halves[:, 1, :]   # all players, second half, all 7 matches

first_half_avg  = first_half.mean(axis=1)
second_half_avg = second_half.mean(axis=1)

print("FORM COMPARISON: First half vs Second half")
print("=" * 55)
print(f"  {'Player':<10} {'1st Half Avg':>13}  {'2nd Half Avg':>13}  {'Trend'}")
print("  " + "-" * 50)
for i, p in enumerate(PLAYERS):
    trend = "Improving" if second_half_avg[i] > first_half_avg[i] else "Declining"
    print(f"  {p:<10} {first_half_avg[i]:>13.1f}  {second_half_avg[i]:>13.1f}  {trend}")

Reshaped to: (5, 2, 7)

FORM COMPARISON: First half vs Second half
  Player      1st Half Avg   2nd Half Avg  Trend
  --------------------------------------------------
  Virat               57.6           53.7  Declining
  Rohit               50.9           50.6  Declining
  Hardik              42.0           40.9  Declining
  Shubman             45.9           46.0  Improving
  Shreyas             51.4           47.9  Declining



# Transposition

## What is a transpose?

**Transposing** an array flips it across its diagonal — rows become columns, and columns become rows. For a 2D array of shape `(r, c)`, the transpose has shape `(c, r)`.

```
Original (players x matches), shape (5, 14):

          M1  M2  M3  M4 ...
Virat  [  72  45  88  31 ...]
Rohit  [  56   0  74  88 ...]
Hardik [  45  32   0  67 ...]

Transposed (matches x players), shape (14, 5):

       Virat  Rohit  Hardik
M1  [   72     56      45  ]
M2  [   45      0      32  ]
M3  [   88     74       0  ]
```

Notice: the **data does not change** — only the way it is organised. The value `72` is still Virat's match 1 score in both versions; it just moved from `[0, 0]` to `[0, 0]` (in this case the same position, but in general the position changes).

In [12]:
# Transposing in NumPy: .T or np.transpose()

print(f"Original shape: {squad_scores.shape}  (players x matches)")
print(squad_scores[:, :4])  # showing first 4 matches only for readability
print()

transposed = squad_scores.T   # the short, common way
# Equivalent: transposed = np.transpose(squad_scores)

print(f"Transposed shape: {transposed.shape}  (matches x players)")
print(transposed[:4, :])  # showing first 4 matches (now rows) only
print()

# Verify the data is unchanged, just reorganised
print(f"Original  squad_scores[1, 2]   (Rohit, match 3) = {squad_scores[1, 2]}")
print(f"Transposed     transposed[2, 1]   (match 3, Rohit) = {transposed[2, 1]}")
print("Same value, accessed with row/column swapped — that is exactly what transpose does.")

Original shape: (5, 14)  (players x matches)
[[72 45 88 31]
 [56  0 74 88]
 [45 32  0 67]
 [33 61 45  0]
 [88 14 57 72]]

Transposed shape: (14, 5)  (matches x players)
[[72 56 45 33 88]
 [45  0 32 61 14]
 [88 74  0 45 57]
 [31 88 67  0 72]]

Original  squad_scores[1, 2]   (Rohit, match 3) = 74
Transposed     transposed[2, 1]   (match 3, Rohit) = 74
Same value, accessed with row/column swapped — that is exactly what transpose does.


## Why would you transpose data?

**Reason 1: Changing your analysis perspective**

If your data is organised as players x matches, but you want to ask match-centric questions ("show me everyone's performance in match 7"), it can be more natural to have matches as rows.

**Reason 2: Matching shapes for an operation**

Some operations — especially matrix multiplication, which we cover next — have strict shape requirements. Transposing is often the tool you use to make two arrays compatible.

**Reason 3: Compatibility with other libraries**

Many libraries (including Pandas, which comes next in this course) expect data oriented a certain way — typically rows as records/observations and columns as variables/features. Transposing lets you adapt your data to match that expectation.

In [13]:
# Practical use: viewing the data "match-first" instead of "player-first"

match_first = squad_scores.T   # shape (14, 5)

print("Match-centric view — each row is one match, each column is one player:")
print()
print(f"  {'Match':<7}" + "".join(f"{p:>10}" for p in PLAYERS))
for i in range(14):
    print(f"  {i+1:<7}" + "".join(f"{match_first[i, j]:>10}" for j in range(5)))

print()
print("This view makes it natural to ask: 'who had the best match 7?'")
best_in_match7 = PLAYERS[match_first[6].argmax()]
print(f"Best performer in Match 7: {best_in_match7} ({match_first[6].max()} runs)")

Match-centric view — each row is one match, each column is one player:

  Match       Virat     Rohit    Hardik   Shubman   Shreyas
  1              72        56        45        33        88
  2              45         0        32        61        14
  3              88        74         0        45        57
  4              31        88        67         0        72
  5              64        32        18        77         0
  6               0        45        54        83        38
  7             103        61        78        22        91
  8              52        17        22        58        43
  9              19        90        41        44        66
  10             77        38        33        71        22
  11             41        72        60         0        84
  12             93         5        15        66         0
  13             28        83        88        31        47
  14             66        49        27        52        73

This view makes it natural 


# — Matrix Multiplication Basics

## First — what is a matrix?

A matrix is just another word for a 2D array — numbers arranged in rows and columns. `squad_scores` is a matrix. So is a spreadsheet, conceptually.

## Element-wise multiply vs matrix multiply — these are NOT the same thing

This is the single most important distinction in this section.

- **Element-wise multiply (`*`)** — what we have used all along. Multiplies corresponding positions. Both arrays must have the **same shape**. The result has the **same shape**.
- **Matrix multiply (`@` or `np.matmul()` or `np.dot()`)** — a completely different mathematical operation, used to combine two matrices in a way that produces something new. The shapes follow different rules, and the result usually has a **different shape**.

 ## how matrix multiplication actually works

Let's work through the mechanics with small, simple numbers before touching any cricket data.

Take two small matrices:

```
      A (shape 2x3)              B (shape 3x2)

    [ 1   2   3 ]              [ 7   8  ]
    [ 4   5   6 ]              [ 9  10  ]
                                [ 11 12  ]
```

**Rule for whether multiplication is even possible:** the number of **columns** in A must equal the number of **rows** in B.

```
A is (2, 3)  ->  3 columns
B is (3, 2)  ->  3 rows
3 == 3  ->  multiplication is allowed
```

**The shape of the result:** take the outer numbers — A's rows and B's columns.

```
A (2, 3) @ B (3, 2)  ->  result shape (2, 2)
        ^         ^
      these must match (inner dimensions)
  ^                  ^
 these become the result shape (outer dimensions)
```

**How each value in the result is computed:** for the result's position `[row, col]`, take **row `row` of A** and **column `col` of B**, multiply them element by element, and sum.

```
result[0, 0] = (row 0 of A) . (column 0 of B)
             = (1*7) + (2*9) + (3*11)
             =   7   +  18   +  33
             =   58

result[0, 1] = (row 0 of A) . (column 1 of B)
             = (1*8) + (2*10) + (3*12)
             =   8   +   20   +   36
             =   64

result[1, 0] = (row 1 of A) . (column 0 of B)
             = (4*7) + (5*9) + (6*11)
             =  28   +  45   +  66
             =  139

result[1, 1] = (row 1 of A) . (column 1 of B)
             = (4*8) + (5*10) + (6*12)
             =  32   +  50    +  72
             =  154
```

```
Result =  [  58   64 ]
          [ 139  154 ]
```

Each result cell is a **dot product**: multiply matching positions, then add them all up. Let's verify this with NumPy.

In [14]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])

B = np.array([[7, 8],
              [9, 10],
              [11, 12]])

print(f"A shape: {A.shape}")
print(f"B shape: {B.shape}")
print()

# Matrix multiplication — the @ operator (preferred, modern syntax)
result = A @ B

print(f"A @ B shape: {result.shape}")
print("A @ B =")
print(result)
print()
print("Matches our hand calculation exactly: [[58, 64], [139, 154]]")
print()

# np.matmul() and np.dot() do the same thing for 2D arrays
result_matmul = np.matmul(A, B)
result_dot    = np.dot(A, B)
print(f"np.matmul(A, B) gives the same result: {(result == result_matmul).all()}")
print(f"np.dot(A, B) gives the same result:    {(result == result_dot).all()}")

A shape: (2, 3)
B shape: (3, 2)

A @ B shape: (2, 2)
A @ B =
[[ 58  64]
 [139 154]]

Matches our hand calculation exactly: [[58, 64], [139, 154]]

np.matmul(A, B) gives the same result: True
np.dot(A, B) gives the same result:    True


In [15]:
a = np.array([
    [1, 2],
    [3, 4]
])

b = np.array([
    [5, 6],
    [7, 8]
])


In [16]:
a @ b

array([[19, 22],
       [43, 50]])

In [17]:
# Clear side-by-side: element-wise * vs matrix @
# Using two SQUARE matrices so element-wise multiply is even possible for comparison

X = np.array([[1, 2],
              [3, 4]])

Y = np.array([[5, 6],
              [7, 8]])

elementwise_result = X * Y     # position-by-position
matrix_result      = X @ Y     # row-by-column dot products

print("X =")
print(X)
print("Y =")
print(Y)
print()
print("X * Y  (element-wise — position by position):")
print(elementwise_result)
print("  [0,0] = 1*5 = 5      [0,1] = 2*6 = 12")
print("  [1,0] = 3*7 = 21     [1,1] = 4*8 = 32")
print()
print("X @ Y  (matrix multiply — row dot column):")
print(matrix_result)
print("  [0,0] = (1*5)+(2*7) = 19      [0,1] = (1*6)+(2*8) = 22")
print("  [1,0] = (3*5)+(4*7) = 43      [1,1] = (3*6)+(4*8) = 50")
print()
print("Completely different results from the same two matrices.")
print("This is why mixing up * and @ is a common and serious bug.")

X =
[[1 2]
 [3 4]]
Y =
[[5 6]
 [7 8]]

X * Y  (element-wise — position by position):
[[ 5 12]
 [21 32]]
  [0,0] = 1*5 = 5      [0,1] = 2*6 = 12
  [1,0] = 3*7 = 21     [1,1] = 4*8 = 32

X @ Y  (matrix multiply — row dot column):
[[19 22]
 [43 50]]
  [0,0] = (1*5)+(2*7) = 19      [0,1] = (1*6)+(2*8) = 22
  [1,0] = (3*5)+(4*7) = 43      [1,1] = (3*6)+(4*8) = 50

Completely different results from the same two matrices.
This is why mixing up * and @ is a common and serious bug.


## A practical use: weighted scoring

Matrix multiplication is everywhere in data science — it is the backbone of linear regression, neural networks, and recommendation systems. Here is a simple, intuitive use: computing a **weighted fantasy score** for every player, in every match, using a single matrix multiplication.

**The scenario:** A fantasy cricket platform scores a player based on multiple stats: runs, wickets, and catches. Each stat has a different point value. We want to compute the total fantasy points for every player in every match — in one operation.

In [18]:
# Stats matrix: for ONE match, 5 players x 3 stat categories (runs, wickets, catches)
match_stats = np.array([
    [72, 0, 1],   # Virat:   72 runs, 0 wickets, 1 catch
    [56, 0, 0],   # Rohit:   56 runs, 0 wickets, 0 catches
    [12, 2, 1],   # Hardik:  12 runs, 2 wickets, 1 catch
    [33, 0, 2],   # Shubman: 33 runs, 0 wickets, 2 catches
    [45, 1, 0],   # Shreyas: 45 runs, 1 wicket,  0 catches
])

# Points per stat category: 1 point per run, 25 points per wicket, 10 points per catch
point_values = np.array([
    [1],     # points per run
    [25],    # points per wicket
    [10],    # points per catch
])

print(f"match_stats shape:  {match_stats.shape}   (5 players x 3 stats)")
print(f"point_values shape: {point_values.shape}    (3 stats x 1 point-value column)")
print()

# Matrix multiply: (5, 3) @ (3, 1) -> (5, 1)
# Each player's fantasy score = (runs*1) + (wickets*25) + (catches*10)
fantasy_points = match_stats @ point_values

print(f"Result shape: {fantasy_points.shape}")
print()
print("FANTASY POINTS")
print("=" * 30)
for i, p in enumerate(PLAYERS):
    print(f"  {p:<10} {fantasy_points[i, 0]:>5} points")

print()
print(f"Check Hardik by hand: (12*1) + (2*25) + (1*10) = {12*1 + 2*25 + 1*10}")
print(f"Matches fantasy_points[2,0] = {fantasy_points[2, 0]}")

match_stats shape:  (5, 3)   (5 players x 3 stats)
point_values shape: (3, 1)    (3 stats x 1 point-value column)

Result shape: (5, 1)

FANTASY POINTS
  Virat         82 points
  Rohit         56 points
  Hardik        72 points
  Shubman       53 points
  Shreyas       70 points

Check Hardik by hand: (12*1) + (2*25) + (1*10) = 72
Matches fantasy_points[2,0] = 72


# Generating Synthetic Data

## Why generate fake data?

Every dataset we have used in this course so far has been hand-written by your instructor. In the real world, you will often need to **generate data yourself**, for reasons like:

- **Testing code** before real data is available
- **Simulating scenarios** — "what if we had 10,000 matches instead of 14?"
- **Learning and prototyping** without needing access to a real, possibly sensitive, dataset
- **Statistical simulation** — modelling random processes (dice rolls, coin flips, customer arrivals, noisy sensor readings)

NumPy's `np.random` module is the standard tool for this.

---

## The most important rule: setting a seed# 

In [19]:
# Without a seed, random numbers are different every time you run the cell

print("Without a seed (run this cell multiple times — values change each time):")
print(np.random.randint(1, 100, size=5))
print(np.random.randint(1, 100, size=5))
print()

# With a seed, the sequence becomes reproducible
print("With a fixed seed (always the same values):")
np.random.seed(42)
print(np.random.randint(1, 100, size=5))

np.random.seed(42)   # reset to the same seed
print(np.random.randint(1, 100, size=5))
print()
print("Setting a seed makes your 'random' results reproducible.")
print("This matters for debugging, sharing results, and grading exercises consistently.")

Without a seed (run this cell multiple times — values change each time):
[85 15 19 50 33]
[37 53  3 29 34]

With a fixed seed (always the same values):
[52 93 15 72 61]
[52 93 15 72 61]

Setting a seed makes your 'random' results reproducible.
This matters for debugging, sharing results, and grading exercises consistently.


## The main random generation functions

| Function | Generates | Example |
|---|---|---|
| `np.random.randint(low, high, size)` | Random integers, `low` to `high-1` | Dice rolls, scores, counts |
| `np.random.rand(size)` | Random floats, uniformly between 0 and 1 | Probabilities, percentages |
| `np.random.uniform(low, high, size)` | Random floats, uniformly between `low` and `high` | Any bounded continuous value |
| `np.random.normal(mean, std, size)` | Random floats following a bell curve | Heights, measurement noise, most natural data |
| `np.random.choice(array, size)` | Random picks FROM an existing array | Sampling from real categories |

In [20]:
np.random.seed(42)

# randint — random integers
dice_rolls = np.random.randint(1, 7, size=10)   # 1 to 6 inclusive
print(f"randint(1, 7, size=10)  — dice rolls: {dice_rolls}")
print()

# rand — uniform floats 0 to 1
probabilities = np.random.rand(5)
print(f"rand(5) — uniform 0 to 1: {probabilities.round(3)}")
print()

# uniform — uniform floats in a custom range
temperatures = np.random.uniform(15, 35, size=5)
print(f"uniform(15, 35, size=5) — temperatures: {temperatures.round(1)}")
print()

# normal — bell-curve distributed floats
heights = np.random.normal(170, 8, size=5)   # mean=170cm, std=8cm
print(f"normal(170, 8, size=5) — heights:    {heights.round(1)}")
print()

# choice — random picks from an existing array
outcomes = np.random.choice(["win", "loss", "tie"], size=8)
print(f"choice(['win','loss','tie'], size=8): {outcomes}")

randint(1, 7, size=10)  — dice rolls: [4 5 3 5 5 2 3 3 3 5]

rand(5) — uniform 0 to 1: [0.601 0.708 0.021 0.97  0.832]

uniform(15, 35, size=5) — temperatures: [19.2 18.6 18.7 21.1 25.5]

normal(170, 8, size=5) — heights:    [156.2 165.5 161.9 172.5 162.7]

choice(['win','loss','tie'], size=8): ['loss' 'loss' 'tie' 'loss' 'tie' 'tie' 'win' 'tie']


### Why `normal` deserves special attention

Most real-world measurements cluster around an average, with values further from the average becoming progressively rarer. This pattern is called a **normal distribution** (or bell curve). Player batting scores, human heights, measurement errors, exam scores — most of these roughly follow this shape.

`np.random.normal(mean, std, size)` takes exactly the two parameters that describe a bell curve:
- `mean` — the centre of the curve (most values cluster here)
- `std` — how spread out the values are (we covered this in Session 5.2 — standard deviation)

This is usually the right choice when simulating realistic human or natural data, rather than `randint` or `uniform`, which spread values evenly and do not reflect how most real things actually behave.